# Automatic Transcription, OCR, and Semantic Search

In this lab, you'll connect to your Snowflake environment, set up your personal workspace, and build a multimodal data pipeline. You'll process audio recordings and slide images to extract searchable text, then use vector embeddings to enable semantic retrieval across both modalities.

Let's begin!

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download this notebook:</b> Click <em>"File"</em> → <em>"Download"</em>.</p>
</div>


## Connect to Snowflake

Load your credentials from the environment and create a Snowflake session. This session is how you'll interact with Snowflake throughout the lab — creating tables, running queries, and calling AI functions.

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> <b>Note:</b> The environment variables referenced in the cell below are already provided in this learning environment, along with a pre-configured Snowflake account. You don't need to set anything up to run this lab.</p>
<p>These variables tell the Snowflake session how and where to connect:</p>
<ul>
<li><code>SNOWFLAKE_ACCOUNT</code>: your Snowflake account identifier</li>
<li><code>SNOWFLAKE_USER</code>: the username connecting to Snowflake</li>
<li><code>SNOWFLAKE_PAT</code>: a programmatic access token used for authentication</li>
<li><code>SNOWFLAKE_ROLE</code>: the role whose privileges the session uses</li>
<li><code>SNOWFLAKE_WAREHOUSE</code>: the compute warehouse that runs your queries</li>
<li><code>SNOWFLAKE_DATABASE</code>: the default database for the session</li>
</ul>
<p>If you'd like to run this notebook locally, you'll need your own Snowflake account. You can <a href="https://signup.snowflake.com/" target="_blank">create a free Snowflake trial account here</a>, then see the <a href="https://docs.snowflake.com/en/user-guide/programmatic-access-tokens" target="_blank">programmatic access tokens guide</a> to generate a PAT and set these variables in your local environment.</p>
</div>

In [ ]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.core import Root, CreateMode
from snowflake.core.schema import Schema
from snowflake.core.table import Table, TableColumn

_ = load_dotenv(override=True)

session = Session.builder.configs({
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "authenticator": "programmatic_access_token",
    "token": os.getenv("SNOWFLAKE_PAT"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
}).create()

root = Root(session)

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
SHARED_SCHEMA = "shared"
INTERNAL_STAGE = f"{SHARED_SCHEMA}.int_multimodal_data_files"

current_user = session.get_current_user()
print(f"Connected as {current_user}")
print(f"Welcome to the course! Let's start building.")

## Create Your Schema

Create a personal schema for your work in this course.

In [ ]:
LEARNER_SCHEMA = current_user.strip('"').lower()

schema = Schema(name=LEARNER_SCHEMA)
root.databases[DATABASE].schemas.create(schema, mode=CreateMode.if_not_exists)
session.use_schema(LEARNER_SCHEMA)

print(f"Using schema: {DATABASE}.{LEARNER_SCHEMA}")

## Transcribe Audio Files

Use `AI_TRANSCRIBE` to run ASR on meeting audio recordings (WAV files) and convert them to text. The transcription results — including the full text and audio duration — are stored in a structured table for downstream processing.

In [ ]:
audio_transcripts_table = Table(
    name="audio_transcripts",
    columns=[
        TableColumn(name="meeting_id", datatype="STRING"),
        TableColumn(name="meeting_part", datatype="STRING"),
        TableColumn(name="audio_file_path", datatype="STRING"),
        TableColumn(name="transcript_text", datatype="STRING"),
        TableColumn(name="audio_duration", datatype="FLOAT"),
    ]
)

root.databases[DATABASE].schemas[LEARNER_SCHEMA].tables.create(
    audio_transcripts_table,
    mode=CreateMode.or_replace
)

print("Created audio_transcripts table")

meeting_id = "ES2008"
meeting_parts = ["a", "b", "c", "d"]

Run the transcription loop over each meeting part, calling `AI_TRANSCRIBE` on the corresponding audio file.

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ &nbsp; <b>This step may take 2–5 minutes</b> as it processes multiple audio files through the ASR service.</p>

In [ ]:
for part in meeting_parts:
    audio_path = f"@{INTERNAL_STAGE}/amicorpus/ES2008{part}/audio/ES2008{part}.Mix-Headset.wav"

    insert_query = f"""
    INSERT INTO audio_transcripts
    SELECT
        '{meeting_id}' AS meeting_id,
        '{part}' AS meeting_part,
        '{audio_path}' AS audio_file_path,
        SNOWFLAKE.CORTEX.AI_TRANSCRIBE(
            TO_FILE('{audio_path}')
        ):text::STRING AS transcript_text,
        SNOWFLAKE.CORTEX.AI_TRANSCRIBE(
            TO_FILE('{audio_path}')
        ):audio_duration::FLOAT AS audio_duration
    """

    session.sql(insert_query).collect()
    print(f"Transcribed audio for meeting {meeting_id}{part}")

Preview the transcription results to verify the text and duration were captured correctly.

In [ ]:
print("Audio Transcription Results:")
df = session.table("audio_transcripts")
df.select(
    F.col("meeting_id"),
    F.col("meeting_part"),
    F.left(F.col("transcript_text"), 100).alias("preview"),
    F.col("audio_duration").alias("audio_duration_secs")
).show()

## Extract Text from Slides (OCR)

Use `AI_PARSE_DOCUMENT` to extract text from slide images (JPG) using OCR. Each slide's extracted text is stored alongside its metadata for later embedding and search.

In [ ]:
slides_ocr_table = Table(
    name="slides_ocr",
    columns=[
        TableColumn(name="meeting_id", datatype="STRING"),
        TableColumn(name="meeting_part", datatype="STRING"),
        TableColumn(name="slide_filename", datatype="STRING"),
        TableColumn(name="slide_file_path", datatype="STRING"),
        TableColumn(name="extracted_text", datatype="STRING"),
    ]
)

root.databases[DATABASE].schemas[LEARNER_SCHEMA].tables.create(
    slides_ocr_table,
    mode=CreateMode.or_replace
)

print("Created slides_ocr table")

Run OCR on each meeting part's slide images and insert the extracted text into the `slides_ocr` table.


<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ &nbsp; <b>This step may take 1–3 minutes</b> as each slide image is processed through the OCR service.</p>

In [ ]:
for part in meeting_parts:
    insert_query = f"""
    INSERT INTO slides_ocr
    SELECT
        '{meeting_id}' AS meeting_id,
        '{part}' AS meeting_part,
        SPLIT_PART(relative_path, '/', -1) AS slide_filename,
        relative_path AS slide_file_path,
        TO_VARCHAR(
            SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
                TO_FILE('@{INTERNAL_STAGE}/' || relative_path),
                OBJECT_CONSTRUCT('mode', 'LAYOUT')
            )
        ) AS extracted_text
    FROM DIRECTORY(@{INTERNAL_STAGE})
    WHERE relative_path LIKE 'amicorpus/ES2008{part}/slides/%.jpg'
    """

    session.sql(insert_query).collect()
    print(f"Extracted text from slides for meeting {meeting_id}{part}")

Preview the OCR results to see the extracted text from each slide.

In [ ]:
print("Slide OCR Results:")
slides_df = session.table("slides_ocr")
slides_df.select(
    "meeting_id", "meeting_part", "slide_filename",
    F.left(F.col("extracted_text"), 100).alias("preview")
).limit(10).show()

for row in slides_df.limit(10).collect():
    print(f"\n{row['SLIDE_FILENAME']}:")
    print(row['EXTRACTED_TEXT'])

## Chunk Audio Transcripts

Long transcripts aren't ideal for semantic search — a single embedding can't capture everything in a multi-minute recording. Here, you'll split each transcript into smaller, overlapping chunks. The overlap ensures that context isn't lost at chunk boundaries.

In [ ]:
transcript_chunks_table = Table(
    name="transcript_chunks",
    columns=[
        TableColumn(name="chunk_id", datatype="STRING"),
        TableColumn(name="meeting_id", datatype="STRING"),
        TableColumn(name="meeting_part", datatype="STRING"),
        TableColumn(name="chunk_index", datatype="INTEGER"),
        TableColumn(name="chunk_text", datatype="STRING"),
    ]
)

root.databases[DATABASE].schemas[LEARNER_SCHEMA].tables.create(
    transcript_chunks_table,
    mode=CreateMode.or_replace
)

print("Created transcript_chunks table")

CHUNK_SIZE = 500
OVERLAP = 50

Loop through each transcript, split it into overlapping chunks, and append the results to the `transcript_chunks` table.

In [ ]:
session.sql("TRUNCATE TABLE IF EXISTS transcript_chunks").collect()

transcripts_df = session.table("audio_transcripts")

for part in meeting_parts:
    rows = transcripts_df.filter(
        F.col("meeting_part") == part
    ).select(
        "meeting_id", "meeting_part", "transcript_text"
    ).collect()

    for row in rows:
        text = row['TRANSCRIPT_TEXT']
        m_id = row['MEETING_ID']
        m_part = row['MEETING_PART']

        chunks = []
        chunk_idx = 0
        i = 0

        while i < len(text):
            chunk = text[i:i + CHUNK_SIZE]
            chunks.append((
                f"{m_id}_{m_part}_{chunk_idx}",
                m_id,
                m_part,
                chunk_idx,
                chunk
            ))
            chunk_idx += 1
            i += CHUNK_SIZE - OVERLAP

        if chunks:
            chunk_df = session.create_dataframe(
                chunks,
                schema=["chunk_id", "meeting_id", "meeting_part", "chunk_index", "chunk_text"]
            )
            chunk_df.write.save_as_table("transcript_chunks", mode="append")

    print(f"Chunked transcript for meeting {meeting_id}{part}")

Preview the chunks and review basic statistics like chunk count and average length per meeting part.

In [ ]:
print("Transcript Chunks Sample:")
chunks_df = session.table("transcript_chunks")
chunks_df.select(
    "chunk_id", "meeting_id", "meeting_part", "chunk_index",
    F.left(F.col("chunk_text"), 100).alias("chunk_preview")
).sort(
    "meeting_id", "meeting_part", "chunk_index"
).limit(10).show()

print("\nChunk Statistics:")
chunks_df.group_by(
    "meeting_id", "meeting_part"
).agg(
    F.count("*").alias("chunk_count"),
    F.avg(F.length(F.col("chunk_text"))).alias("avg_chunk_length")
).sort(
    "meeting_id", "meeting_part"
).show()

## Generate Vector Embeddings

Convert the text content into vector embeddings using an embedding model (`snowflake-arctic-embed-m`). These embeddings capture the *meaning* of each piece of text as a high-dimensional vector, enabling you to search by semantic similarity rather than exact keyword matches.

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ &nbsp; <b>This step may take 1–2 minutes</b> as embeddings are generated for all transcript chunks and slide text.</p>

In [ ]:
# 768 dimensions to match the snowflake-arctic-embed-m output
session.sql("""
    ALTER TABLE transcript_chunks
    ADD COLUMN text_embedding VECTOR(FLOAT, 768)
""").collect()

session.sql("""
    UPDATE transcript_chunks
    SET text_embedding = AI_EMBED(
        'snowflake-arctic-embed-m',
        chunk_text
    )
    WHERE chunk_text IS NOT NULL
""").collect()

print("Generated embeddings for transcript chunks")

session.sql("""
    ALTER TABLE slides_ocr
    ADD COLUMN text_embedding VECTOR(FLOAT, 768)
""").collect()

session.sql("""
    UPDATE slides_ocr
    SET text_embedding = AI_EMBED(
        'snowflake-arctic-embed-m',
        extracted_text
    )
    WHERE extracted_text IS NOT NULL
""").collect()

print("Generated embeddings for slide OCR text")

Verify that embeddings were generated by previewing a few transcript chunks with their embedding vectors.

In [ ]:
print("Embedding Samples:")
chunks_df = session.table("transcript_chunks")
chunks_df.filter(
    F.col("text_embedding").is_not_null()
).select(
    "chunk_id", "meeting_id", "meeting_part",
    F.left(F.col("chunk_text"), 100).alias("chunk_preview"),
    F.col("text_embedding").alias("embedding_vector")
).limit(5).show()

## Semantic Search: Audio Chunks

Now that your text is embedded, you can search by *meaning*. Enter a search phrase, and the system will find the audio chunks whose content is most semantically similar – even if they don't contain the exact words you searched for.

<div style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px">
<p>🔍 &nbsp; <b>Results may vary:</b> Semantic search results depend on the embedding model and the content of your transcriptions. Your results may differ slightly from those shown in the video.</p>
</div>

In [ ]:
# Semantic Search - find similar audio chunks

# Create embedding for search query
search_query = "budget for the remote control design"

search_results = session.sql(f"""
    WITH query_embedding AS (
        SELECT AI_EMBED(
            'snowflake-arctic-embed-m',
            '{search_query}'
        ) AS embedding
    )
    SELECT
        c.chunk_id,
        c.meeting_id,
        c.meeting_part,
        c.chunk_index,
        LEFT(c.chunk_text, 500) AS chunk_preview,
        VECTOR_COSINE_SIMILARITY(c.text_embedding, q.embedding) AS similarity_score
    FROM transcript_chunks c, query_embedding q
    WHERE c.text_embedding IS NOT NULL
    ORDER BY similarity_score DESC
    LIMIT 5
""")

search_results.show()


results = search_results.collect()
print(results[0]['CHUNK_PREVIEW'])

## Semantic Search: Slides

Search across slide content using the same semantic approach. This lets you find relevant slides even when the exact terminology differs from your search phrase.

In [ ]:
# Semantic Search - Find relevant slides

# Search for slides related to design
slides_search_query = "product features about the remote control"

slides_search_results = session.sql(f"""
    WITH query_embedding AS (
        SELECT AI_EMBED(
            'snowflake-arctic-embed-m',
            '{slides_search_query}'
        ) AS embedding
    )
    SELECT
        s.meeting_id,
        s.meeting_part,
        s.slide_filename,
        LEFT(s.extracted_text, 500) AS text_preview,
        VECTOR_COSINE_SIMILARITY(s.text_embedding, q.embedding) AS similarity_score
    FROM slides_ocr s, query_embedding q
    WHERE s.text_embedding IS NOT NULL
    ORDER BY similarity_score DESC
    LIMIT 5
""")

slides_search_results.show()

results = slides_search_results.collect()
print(results[0]['TEXT_PREVIEW'])

## Cross-Modal Search

Search across **both** audio transcripts and slide content simultaneously. This is the power of a multimodal pipeline — a single query can surface related information from completely different data sources.

In [ ]:
cross_modal_search_query = "product features about the remote control"

print(f"Cross-modal search: '{cross_modal_search_query}'")

session.sql(f"""
        WITH query_embedding AS (
            SELECT AI_EMBED(
                'snowflake-arctic-embed-m',
                '{cross_modal_search_query}'
            ) AS embedding
        ),
        combined AS (
            SELECT DISTINCT
                'AUDIO' AS content_type,
                c.meeting_id,
                c.meeting_part,
                LEFT(c.chunk_text, 60) AS content_preview,
                
  ROUND(VECTOR_COSINE_SIMILARITY(c.text_embedding,
  q.embedding), 2) AS similarity
            FROM transcript_chunks c, query_embedding q
            WHERE c.text_embedding IS NOT NULL
            ORDER BY similarity DESC
            LIMIT 5
        ),
        combined_all AS (
            SELECT * FROM combined
            UNION ALL
            SELECT DISTINCT
                'SLIDE' AS content_type,
                s.meeting_id,
                s.meeting_part,
                LEFT(s.extracted_text, 60) AS 
  content_preview,
                
  ROUND(VECTOR_COSINE_SIMILARITY(s.text_embedding,
  q.embedding), 2) AS similarity
            FROM slides_ocr s, query_embedding q
            WHERE s.text_embedding IS NOT NULL
            ORDER BY similarity DESC
            LIMIT 10
        )
        SELECT * FROM combined_all
        ORDER BY similarity DESC
    """).show()

## Next Steps

In this lab, you:

1. **Connected to Snowflake** — Created a session and set up a personal schema for your work
2. **Transcribed meeting audio** — Used `AI_TRANSCRIBE` to convert WAV recordings into text with duration metadata
3. **Extracted text from slides** — Ran OCR on slide images with `AI_PARSE_DOCUMENT` in `LAYOUT` mode
4. **Chunked transcripts** — Split long transcripts into overlapping chunks so each embedding captures a focused span of meaning
5. **Generated vector embeddings** — Embedded transcript chunks and slide text with `snowflake-arctic-embed-m` into 768-dimensional vectors
6. **Semantic search on audio** — Found transcript chunks by meaning using `VECTOR_COSINE_SIMILARITY`
7. **Semantic search on slides** — Surfaced relevant slides even when the wording differed from the query
8. **Cross-modal search** — Queried audio and slide embeddings together from a single search phrase

Two modalities now have vector embeddings ready for retrieval:
- Audio transcript chunks (`transcript_chunks.text_embedding`)
- Slide OCR text (`slides_ocr.text_embedding`)

In the next labs, you'll extend this pipeline to a third modality — **video** — by processing meeting recordings with a Vision Language Model and combining all three sources into a unified multimodal search and RAG experience.

## References

Snowflake resources covering the pieces introduced in this lab:

- [Cortex AI Functions: Audio](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-audio) - Guide for audio AI functions, including `AI_TRANSCRIBE` used here for ASR on the meeting WAV files
- [Cortex AI Functions: Documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-documents) - Guide for document AI functions, including `AI_PARSE_DOCUMENT` used here for OCR on slide images in `LAYOUT` mode
- [AI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_embed) - SQL reference for the embedding function used to turn transcript chunks and slide text into 768-dimensional vectors with `snowflake-arctic-embed-m`